In [1]:
# A3-1 setup
import os, json
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import os; os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # guards against the torch/numpy OpenMP clash on Windows

root = Path.cwd()
while not (root / "data" / "imgs_cleaned").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
CLEAN = Path("data/imgs_cleaned"); A1 = Path("logs/preprocess_a1")
META = Path("data/meta"); OUT = Path("logs/preprocess_a3"); (OUT / "sheets").mkdir(parents=True, exist_ok=True)

m = pd.read_csv(A1 / "measurements.csv")[["file", "cls", "source", "w", "h"]]
rv = pd.read_csv(META / "review.csv")
drop = set(rv.loc[rv.action == "drop", "file"])
cur = dict(zip(rv.loc[rv.action == "relabel", "file"], rv.loc[rv.action == "relabel", "detail"]))
m["label"] = [cur.get(f, c) for f, c in zip(m["file"], m["cls"])]      # relabel applied
m = m[~m["file"].isin(drop)].sort_values("file").reset_index(drop=True)
crop = {r.file: (r.x0, r.y0, r.x1, r.y1) for r in pd.read_csv(A1 / "letterbox.csv").itertuples()}
print(len(m), "images (", len(drop), "dropped,", len(cur), "relabelled,", len(crop), "get letterbox crop)")

def load(f):
    im = Image.open(CLEAN / f).convert("RGB")
    return im.crop(crop[f]) if f in crop else im

1357 images ( 14 dropped, 1 relabelled, 75 get letterbox crop)


In [2]:
# A3-2b load saved embeddings (use this instead of re-running A3-2 after a restart)
E = np.load(META / "emb_resnet50.npy"); Ef = np.load(META / "emb_resnet50_flip.npy")
files = json.load(open(META / "emb_files.json"))
assert files == m["file"].tolist() and E.shape == (len(files), 2048), \
    "embeddings do not match the current image list - rerun A3-2 once, then restart the kernel again"
print(E.shape, E.dtype)

(1357, 2048) float32


In [ ]:
# A3-2 embeddings (ResNet-50, ImageNet weights), original + horizontally flipped view
import torch, torchvision
import torchvision.transforms as T

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
net = torchvision.models.resnet50(weights=w); net.fc = torch.nn.Identity(); net = net.eval().to(dev)
tf = T.Compose([T.Resize((224, 224)), T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

E, Ef = [], []
files = m["file"].tolist()
with torch.inference_mode():
    for i in tqdm(range(0, len(files), 32)):
        x = torch.stack([tf(load(f)) for f in files[i:i + 32]]).to(dev)
        E.append(net(x).cpu()); Ef.append(net(torch.flip(x, dims=[3])).cpu())
E = torch.nn.functional.normalize(torch.cat(E), dim=1).numpy()
Ef = torch.nn.functional.normalize(torch.cat(Ef), dim=1).numpy()
np.save(META / "emb_resnet50.npy", E); np.save(META / "emb_resnet50_flip.npy", Ef)
json.dump(files, open(META / "emb_files.json", "w"))
print(E.shape)

In [3]:
# A3-3 similarity + neighbour stats + sample pairs per similarity band
S = np.maximum(E @ E.T, E @ Ef.T)                # a mirrored copy still counts as a match
S = np.maximum(S, S.T)
np.fill_diagonal(S, -1)
m["nn_sim"] = S.max(axis=1)
m["nn_file"] = [files[j] for j in S.argmax(axis=1)]
print(m.groupby("label").nn_sim.describe(percentiles=[.5]).round(2)[["count", "min", "50%", "max"]])
print("\nshare of images whose nearest neighbour is above:")
print(pd.DataFrame({t: m.groupby("label").nn_sim.apply(lambda s, t=t: (s >= t).mean())
                    for t in (0.95, 0.90, 0.85, 0.80)}).round(2))

iu = np.triu_indices(len(files), 1)
sims = S[iu]
BANDS = [(0.97, 1.01), (0.93, 0.97), (0.88, 0.93), (0.83, 0.88), (0.78, 0.83)]
rng = np.random.default_rng(42)

def pair_sheet(pairs, path, cols=4, th=160):
    nr = -(-len(pairs) // cols)
    sheet = Image.new("RGB", (cols * 2 * th, nr * (th + 14)), "white"); dr = ImageDraw.Draw(sheet)
    for k, (a, b, s) in enumerate(pairs):
        x, y = (k % cols) * 2 * th, (k // cols) * (th + 14)
        for j, f in enumerate((a, b)):
            im = load(f); im.thumbnail((th - 4, th - 4)); sheet.paste(im, (x + j * th, y))
        dr.text((x + 2, y + th), f"{a.split('/')[-1][:-4]} | {b.split('/')[-1][:-4]}  {s:.2f}", fill="red")
    sheet.save(path, quality=90)

for p in (OUT / "sheets").glob("band_*.jpg"): p.unlink()
for lo, hi in BANDS:
    idx = np.where((sims >= lo) & (sims < hi))[0]
    print(f"band {lo:.2f}-{min(hi, 1):.2f}: {len(idx)} pairs")
    if len(idx):
        pick = rng.choice(idx, size=min(24, len(idx)), replace=False)
        pairs = [(files[iu[0][i]], files[iu[1][i]], sims[i]) for i in pick]
        pair_sheet(pairs, OUT / "sheets" / f"band_{lo:.2f}-{min(hi, 1):.2f}.jpg")

                count   min   50%   max
label                                  
dosa            176.0  0.53  0.74  0.97
hoppers         156.0  0.55  0.79  0.99
idli             85.0  0.52  0.70  0.91
kottu            66.0  0.62  0.77  0.90
milk_rice       143.0  0.49  0.83  0.99
paratha         114.0  0.53  0.73  0.95
pittu           246.0  0.50  0.78  0.98
rice            288.0  0.49  0.85  0.97
string_hoppers   83.0  0.58  0.75  0.94

share of images whose nearest neighbour is above:
                0.95  0.90  0.85  0.80
label                                 
dosa            0.03  0.08  0.13  0.26
hoppers         0.35  0.44  0.46  0.49
idli            0.00  0.02  0.02  0.05
kottu           0.00  0.03  0.18  0.39
milk_rice       0.32  0.45  0.49  0.55
paratha         0.04  0.11  0.22  0.25
pittu           0.02  0.12  0.28  0.41
rice            0.16  0.41  0.52  0.70
string_hoppers  0.00  0.06  0.11  0.16
band 0.97-1.00: 46 pairs
band 0.93-0.97: 167 pairs
band 0.88-0.93: 830 pairs
ban

In [4]:
# A3-4 groups: connected components above THR  (set THR after looking at the band sheets)
THR = 0.85

parent = list(range(len(files)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for i, j in zip(*np.where(np.triu(S >= THR, 1))):
    ri, rj = find(i), find(j)
    if ri != rj: parent[rj] = ri
m["group"] = pd.factorize(pd.Series([find(i) for i in range(len(files))]))[0]
m["group_size"] = m.groupby("group")["file"].transform("size")
m["group_classes"] = m.groupby("group")["label"].transform("nunique")
m["group_id"] = m["group"].map(lambda g: f"g{g:04d}")

sz = m.groupby("group").size()
print(f"THR={THR}: {len(sz)} groups for {len(m)} images | singletons: {(sz == 1).sum()} | groups of 2+: {(sz > 1).sum()}")
print(f"images sitting in a group of 2+: {(m.group_size > 1).sum()} ({(m.group_size > 1).mean():.0%})\n")
print(m.groupby("label").agg(images=("file", "size"), groups=("group", "nunique"),
                             in_groups=("group_size", lambda s: int((s > 1).sum())),
                             biggest=("group_size", "max")).to_string())
big = (m.groupby("group").agg(size=("file", "size"), classes=("label", lambda s: ",".join(sorted(set(s)))),
                              first=("file", "min")).sort_values("size", ascending=False).head(10))
print("\n10 largest groups:"); print(big.to_string())
mixed = m[m.group_classes > 1]
print("\ngroups spanning more than one class:", mixed["group"].nunique())

def group_sheet(g, path, th=120, cols=10, maxn=60):
    fs = m.loc[m.group == g, "file"].tolist()[:maxn]
    nr = -(-len(fs) // cols)
    sheet = Image.new("RGB", (cols * th, nr * (th + 12)), "white"); dr = ImageDraw.Draw(sheet)
    for k, f in enumerate(fs):
        im = load(f); im.thumbnail((th - 2, th - 2))
        x, y = (k % cols) * th, (k // cols) * (th + 12)
        sheet.paste(im, (x, y)); dr.text((x + 2, y + th), f.split("/")[-1][:-4], fill="black")
    sheet.save(path, quality=88)

for p in (OUT / "sheets").glob("group_*.jpg"): p.unlink()
for g in big.index[:6]: group_sheet(g, OUT / "sheets" / f"group_big_g{g:04d}.jpg")
for g in mixed["group"].unique()[:12]: group_sheet(g, OUT / "sheets" / f"group_mixedclass_g{g:04d}.jpg")
m[["file", "label", "source", "group_id", "group_size", "group_classes", "nn_sim", "nn_file"]].to_csv(
    META / "near_dup_groups.csv", index=False)
print("saved", META / "near_dup_groups.csv", "| sheets in", OUT / "sheets")

THR=0.85: 1067 groups for 1357 images | singletons: 925 | groups of 2+: 142
images sitting in a group of 2+: 432 (32%)

                images  groups  in_groups  biggest
label                                             
dosa               176     163         23        3
hoppers            156     117         71        3
idli                85      84          2        2
kottu               66      59         12        3
milk_rice          143     106         70        3
paratha            114     101         25        3
pittu              246     205         70        6
rice               288     154        150       49
string_hoppers      83      78          9        3

10 largest groups:
       size classes                 first
group                                    
845      49    rice    rice/rice_0027.jpg
886      28    rice    rice/rice_0128.jpg
862      23    rice    rice/rice_0098.jpg
844      16    rice    rice/rice_0011.jpg
843       8    rice    rice/rice_0009.jpg
661  